<a href="https://colab.research.google.com/github/pviktorr/Projeto_Extensao/blob/main/Projeto_AWS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🚀 Projeto Integrador: Análise de Sentimentos em Avaliações
  
  
  
  Este projeto tem como objetivo realizar a análise de sentimentos em avaliações de clientes, aplicando técnicas de limpeza de dados e processamento de linguagem natural (PLN).

#1. Configuração e Importação de Ferramentas
Nesta etapa, carregamos as bibliotecas necessárias para manipulação de dados, processamento de texto e visualização.

In [ ]:
# importando as ferramentas



import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from textblob import TextBlob
import unicodedata
import re


!pip install emoji
import emoji

#2. Carregamento da Base de Dados
Montamos o Google Drive para acessar os arquivos do projeto e carregamos os datasets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#bancos de dados separados


df_item=pd.read_csv("/content/drive/MyDrive/Projeto Integrador/itens_pedidos.csv")
df_clientes=pd.read_csv("/content/drive/MyDrive/Projeto Integrador/clientes.csv")
df_pagamentos=pd.read_csv("/content/drive/MyDrive/Projeto Integrador/pagamentos.csv")
df_pedidos = pd.read_csv("/content/drive/MyDrive/Projeto Integrador/pedidos.csv")
df_tabela = pd.read_csv("/content/drive/MyDrive/Projeto Integrador/tabela_auxiliar.csv")
df_vendedores = pd.read_csv("/content/drive/MyDrive/Projeto Integrador/vendedores.csv")
df_geolocalizacao = pd.read_csv("/content/drive/MyDrive/Projeto Integrador/geolocalizacao.csv")
df_produtos = pd.read_csv("/content/drive/MyDrive/Projeto Integrador/produtos.csv")
df_avaliacoes = pd.read_csv("/content/drive/MyDrive/Projeto Integrador/avaliacoes.csv")

#3. Limpeza e Tratamento de Dados
Definimos uma função de limpeza para garantir que apenas linhas essenciais (com chaves primárias) sejam removidas, preservando a integridade do dataset.


In [ ]:
#limpeza

def limpar_dados_seguro(df, chaves_obrigatorias):

    df = df.drop_duplicates()
    df = df.dropna(subset=chaves_obrigatorias)
    return df

df_produtos = df_produtos.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

df_clientes = limpar_dados_seguro(df_clientes, ['customer_id'])
df_pedidos = limpar_dados_seguro(df_pedidos, ['order_id', 'customer_id'])
df_item = limpar_dados_seguro(df_item, ['order_id', 'product_id'])
df_produtos = limpar_dados_seguro(df_produtos, ['product_id'])
df_pagamentos = limpar_dados_seguro(df_pagamentos, ['order_id'])
df_vendedores = limpar_dados_seguro(df_vendedores, ['seller_id'])
df_geolocalizacao = limpar_dados_seguro(df_geolocalizacao, ['geolocation_zip_code_prefix'])

df_tabela = df_tabela.drop_duplicates().dropna()

print(f"Limpeza inteligente concluída.")
print(f"Linhas em df_geolocalizacao: {len(df_geolocalizacao)}")
print(f"Linhas em df_produtos: {len(df_produtos)}")
print(f"Linhas em df_pedidos: {len(df_pedidos)}")

In [ ]:
# alteração da definição do dado, ou seja, o tipo dele



colunas_data_pedidos = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in colunas_data_pedidos:
    df_pedidos[col] = pd.to_datetime(df_pedidos[col])

df_item['shipping_limit_date'] = pd.to_datetime(df_item['shipping_limit_date'])

df_clientes['customer_city'] = df_clientes['customer_city'].astype('category')
df_clientes['customer_state'] = df_clientes['customer_state'].astype('category')

df_pedidos['order_status'] = df_pedidos['order_status'].astype('category')

df_pagamentos['payment_type'] = df_pagamentos['payment_type'].astype('category')

df_produtos['product_category_name'] = df_produtos['product_category_name'].astype('category')

df_vendedores['seller_city'] = df_vendedores['seller_city'].astype('category')
df_vendedores['seller_state'] = df_vendedores['seller_state'].astype('category')

df_geolocalizacao['geolocation_city'] = df_geolocalizacao['geolocation_city'].astype('category')
df_geolocalizacao['geolocation_state'] = df_geolocalizacao['geolocation_state'].astype('category')

#4. Análise de Sentimento: Preparação do Texto
Aqui, consolidamos o título e a mensagem da avaliação em uma única coluna, removendo ruídos.

In [ ]:
df_avaliacoes["review_comment_title"] = df_avaliacoes["review_comment_title"].fillna("")
df_avaliacoes["review_comment_message"] = df_avaliacoes["review_comment_message"].fillna("")

df_avaliacoes["review_text"] = ( df_avaliacoes["review_comment_title"].str.strip() + " " + df_avaliacoes["review_comment_message"].str.strip())

df_avaliacoes["review_text"] = (df_avaliacoes["review_text"].str.strip().str.replace(r"\s+", " ", regex=True))

df_avaliacoes["review_text"] = df_avaliacoes["review_text"].replace("", pd.NA)

avaliacoes_limpo = (df_avaliacoes[["review_score", "review_text"]].dropna(subset=["review_text"]).reset_index(drop=True))

In [ ]:
#limpeza texto

def limpar_texto_profissional(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return pd.NA


    texto = unicodedata.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('utf-8')


    texto = re.sub(r'[^a-zA-Z\s]', '', texto)


    return re.sub(r'\s+', ' ', texto).lower().strip()

avaliacoes_limpo["review_text_clean"] = avaliacoes_limpo["review_text"].apply(limpar_texto_profissional)

avaliacoes_limpo = avaliacoes_limpo.dropna(subset=["review_text_clean"]).reset_index(drop=True)

print("Texto limpo com máxima performance!")

#5. Aplicação do VADER para Análise de Sentimento
Utilizamos o algoritmo VADER (Valence Aware Dictionary and sEntiment Reasoner) para classificar o sentimento das avaliações em 5 níveis.

In [ ]:
# istalando o vader

!pip install vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader_en = SentimentIntensityAnalyzer()

def extrair_score_vader_original(texto):
    if pd.isna(texto):
        return 0.0
    return vader_en.polarity_scores(texto)['compound']

def classificar_sentimento_5_niveis(score):

    if score >= 0.6:
        return 'Muito Positivo'
    elif score >= 0.2:
        return 'Positivo'
    elif score > -0.2:
        return 'Neutro'
    elif score > -0.6:
        return 'Negativo'
    else:
        return 'Muito Negativo'

avaliacoes_limpo['vader_score_original'] = avaliacoes_limpo['review_text_clean'].apply(extrair_score_vader_original)

avaliacoes_limpo['sentimento_vader_original'] = avaliacoes_limpo['vader_score_original'].apply(classificar_sentimento_5_niveis)

display(avaliacoes_limpo[['review_score', 'review_text_clean', 'vader_score_original', 'sentimento_vader_original']].head(15))

In [ ]:
contagem_vader = avaliacoes_limpo['sentimento_vader_original'].value_counts()

percentual_vader = avaliacoes_limpo['sentimento_vader_original'].value_counts(normalize=True) * 100

print("=== CONTAGEM ABSOLUTA (VADER ORIGINAL) ===")
print(contagem_vader)
print("\n=== PERCENTUAL (%) ===")
print(percentual_vader.round(2))

#### Visualização da Distribuição de Sentimentos (VADER)

O gráfico de barras abaixo reforça a observação de que o VADER classifica uma parcela muito grande dos textos como 'Neutro', com pouquíssimas ocorrências nas categorias 'Muito Positivo' e 'Muito Negativo'. Isso sugere que o léxico em inglês do VADER não captura adequadamente as nuances e a intensidade do sentimento em português.

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))

ordem_sentimentos = ['Muito Negativo', 'Negativo', 'Neutro', 'Positivo', 'Muito Positivo']

ax = sns.countplot(
    data=avaliacoes_limpo,
    x='sentimento_vader_original',
    order=ordem_sentimentos,
    palette='coolwarm_r'
)

plt.title('Falha de Classificação: VADER Original em Textos Brasileiros', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Sentimento Detectado', fontsize=12)
plt.ylabel('Quantidade de Avaliações', fontsize=12)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom',
                fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### Análise de Sentimento com VADER (Original - Inglês)

O VADER (Valence Aware Dictionary and sEntiment Reasoner) é um algoritmo de análise de sentimento que utiliza um léxico e regras baseadas em heurísticas para classificar textos. Ele é otimizado para o inglês e retorna uma pontuação 'compound' entre -1 (mais negativo) e +1 (mais positivo).

Para esta análise, aplicamos o VADER à coluna `review_text_clean` e classificamos as pontuações em 5 níveis de sentimento: 'Muito Negativo', 'Negativo', 'Neutro', 'Positivo' e 'Muito Positivo'.

#### Resultados do VADER (Contagem e Percentual)

Ao analisar a distribuição dos sentimentos classificados pelo VADER, observamos uma forte concentração na categoria 'Neutro', o que indica uma limitação do modelo ao lidar com o português brasileiro. A maioria dos textos foi considerada neutra, mesmo que pudessem ter uma polaridade clara.

In [ ]:
matriz_confusao = pd.crosstab(
    avaliacoes_limpo['review_score'],
    avaliacoes_limpo['sentimento_vader_original']
)

ordem = ['Muito Negativo', 'Negativo', 'Neutro', 'Positivo', 'Muito Positivo']
matriz_confusao = matriz_confusao.reindex(columns=ordem, fill_value=0)

plt.figure(figsize=(10, 7))
sns.heatmap(matriz_confusao, annot=True, fmt='d', cmap='Reds', cbar=False)

plt.title('Prova de Fogo: Estrelas Reais vs. Previsão do VADER (Inglês)', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('O que o VADER Entendeu', fontsize=12)
plt.ylabel('Estrelas Dadas pelo Cliente (1 a 5)', fontsize=12)

plt.tight_layout()
plt.show()

#### Prova de Fogo: Estrelas Reais vs. Previsão do VADER (Mapa de Calor)

Este mapa de calor compara as estrelas (avaliações numéricas) dadas pelos clientes com as previsões de sentimento do VADER. Idealmente, avaliações com 1 estrela deveriam ser 'Muito Negativas', e avaliações com 5 estrelas deveriam ser 'Muito Positivas'.

No entanto, o mapa mostra que, mesmo para avaliações de 1 e 2 estrelas (claramente negativas), a maioria é classificada como 'Neutro' pelo VADER. Da mesma forma, avaliações de 4 e 5 estrelas (claramente positivas) também têm uma alta frequência de classificação 'Neutro'. Esta visualização evidencia a ineficácia do VADER original para o português brasileiro, que não consegue alinhar suas previsões com o sentimento real expresso nas avaliações.

In [ ]:
plt.figure(figsize=(10, 6))


sns.histplot(avaliacoes_limpo['vader_score_original'], bins=20, kde=True, color='red')

plt.title('Distribuição da Pontuação de Sentimento: VADER Original', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Pontuação Matemática do Sentimento (-1.0 a +1.0)', fontsize=12)
plt.ylabel('Volume de Avaliações', fontsize=12)
plt.grid(axis='y', alpha=0.75)

plt.tight_layout()
plt.show()

#### Distribuição da Pontuação de Sentimento (VADER Original)

O histograma da pontuação 'compound' do VADER revela uma concentração massiva de scores próximos a zero. Isso significa que, para a maioria dos textos em português, o VADER não consegue identificar uma polaridade forte, resultando em pontuações neutras. As poucas pontuações mais extremas (-1 ou +1) são raras, confirmando a dificuldade do modelo em lidar com a língua portuguesa.

### Análise de Sentimento com LeIA (PT-BR)

O LeIA (Léxico para Inferência de Avaliações) é uma biblioteca de análise de sentimento desenvolvida especificamente para o português brasileiro. Diferente do VADER, que é otimizado para o inglês, o LeIA foi treinado com textos em português, o que o torna mais eficaz na compreensão das nuances, gírias e da estrutura da língua, fornecendo classificações de sentimento mais precisas.

In [ ]:
!pip install leia-br

In [ ]:
from LeIA import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def calcular_sentimento_leia(texto):
    if not isinstance(texto, str) or texto.strip() == '':
        return 0.0
    return sia.polarity_scores(texto)['compound']

def classificar_estrelas(score):
    if score >= 0.6: return 'Muito Positivo'
    elif score >= 0.2: return 'Positivo'
    elif score > -0.2: return 'Neutro'
    elif score > -0.6: return 'Negativo'
    else: return 'Muito Negativo'



df_avaliacoes['pontuacao_sentimento_leia'] = df_avaliacoes['review_comment_message'].apply(calcular_sentimento_leia)

df_avaliacoes['classificacao_sentimento'] = df_avaliacoes['pontuacao_sentimento_leia'].apply(classificar_estrelas)

display(df_avaliacoes[['review_score', 'review_comment_message', 'pontuacao_sentimento_leia', 'classificacao_sentimento']].head(20))

In [ ]:
from LeIA import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def calcular_sentimento_leia(texto):
    if not isinstance(texto, str) or texto.strip() == '':
        return 0.0
    return sia.polarity_scores(texto)['compound']

def classificar_estrelas(score):
    if score >= 0.6: return 'Muito Positivo'
    elif score >= 0.2: return 'Positivo'
    elif score > -0.2: return 'Neutro'
    elif score > -0.6: return 'Negativo'
    else: return 'Muito Negativo'

avaliacoes_limpo['pontuacao_sentimento_leia'] = avaliacoes_limpo['review_text_clean'].apply(calcular_sentimento_leia)
avaliacoes_limpo['sentimento_leia'] = avaliacoes_limpo['pontuacao_sentimento_leia'].apply(classificar_estrelas)

contagem_leia = avaliacoes_limpo['sentimento_leia'].value_counts()

percentual_leia = avaliacoes_limpo['sentimento_leia'].value_counts(normalize=True) * 100

print("=== CONTAGEM ABSOLUTA (LeIA - PT-BR) ===")
print(contagem_leia)
print("\n=== PERCENTAGEM (%) ===")
print(percentual_leia.round(2))

#### Resultados do LeIA (Contagem e Percentual)

Ao contrário do VADER, a distribuição de sentimentos do LeIA é muito mais equilibrada e condizente com a realidade. Observamos uma boa distribuição entre as categorias 'Positivo', 'Neutro' e 'Muito Positivo', e uma presença significativa de 'Negativo' e 'Muito Negativo'. Isso demonstra que o LeIA consegue identificar a polaridade de forma mais assertiva em português.

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))

ordem_sentimentos = ['Muito Negativo', 'Negativo', 'Neutro', 'Positivo', 'Muito Positivo']

ax = sns.countplot(
    data=avaliacoes_limpo,
    x='sentimento_leia',
    order=ordem_sentimentos,
    palette='coolwarm_r'
)

plt.title('Distribuição Real de Sentimentos: Algoritmo LeIA (PT-BR)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Sentimento Detectado', fontsize=12)
plt.ylabel('Quantidade de Avaliações', fontsize=12)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom',
                fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

#### Visualização da Distribuição de Sentimentos (LeIA)

O gráfico de barras abaixo ilustra claramente a capacidade do LeIA de classificar o sentimento de forma mais granular e precisa em português. Vemos uma distribuição mais natural e menos enviesada para a categoria 'Neutro', com as avaliações se espalhando pelas cinco categorias de sentimento. Isso reflete a eficácia do modelo em identificar diferentes níveis de polaridade.

In [ ]:
matriz_confusao_leia = pd.crosstab(
    avaliacoes_limpo['review_score'],
    avaliacoes_limpo['sentimento_leia']
)

ordem = ['Muito Negativo', 'Negativo', 'Neutro', 'Positivo', 'Muito Positivo']
matriz_confusao_leia = matriz_confusao_leia.reindex(columns=ordem, fill_value=0)

plt.figure(figsize=(10, 7))
sns.heatmap(matriz_confusao_leia, annot=True, fmt='d', cmap='Greens', cbar=False)

plt.title('Prova de Fogo: Estrelas Reais vs. Previsão do LeIA (PT-BR)', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('O que o LeIA Entendeu', fontsize=12)
plt.ylabel('Estrelas Dadas pelo Cliente (1 a 5)', fontsize=12)

plt.tight_layout()
plt.show()

#### Prova de Fogo: Estrelas Reais vs. Previsão do LeIA (Mapa de Calor)

O mapa de calor do LeIA demonstra uma correlação muito mais forte e lógica entre as estrelas dadas pelos clientes e as classificações de sentimento do modelo. Podemos ver que avaliações de 1 e 2 estrelas estão predominantemente alinhadas com 'Muito Negativo' e 'Negativo', enquanto avaliações de 4 e 5 estrelas tendem a ser classificadas como 'Positivo' ou 'Muito Positivo'. A concentração na diagonal principal mostra que o LeIA está mais em sintonia com o sentimento real dos usuários brasileiros.

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(avaliacoes_limpo['pontuacao_sentimento_leia'], bins=20, kde=True, color='green')

plt.title('Distribuição da Pontuação de Sentimento: LeIA (PT-BR)', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Pontuação Matemática do Sentimento (-1.0 a +1.0)', fontsize=12)
plt.ylabel('Volume de Avaliações', fontsize=12)
plt.grid(axis='y', alpha=0.75)

plt.tight_layout()
plt.show()

#### Distribuição da Pontuação de Sentimento (LeIA PT-BR)

O histograma da pontuação 'compound' do LeIA (PT-BR) mostra uma distribuição mais dispersa e menos concentrada no zero do que a do VADER. Isso indica que o LeIA é capaz de atribuir pontuações mais variadas, refletindo uma maior sensibilidade às nuances do texto em português. Vemos picos nas pontuações positivas e negativas, com uma base menor em pontuações neutras, o que é um sinal de maior acurácia para a língua alvo.

### 8. Análise de Sentimento com Transformer (Pysentimiento)

Utilizamos o `pysentimiento`, uma biblioteca baseada em modelos Transformer pré-treinados para análise de sentimento em português. Esses modelos são geralmente mais robustos e capturam contextos complexos da linguagem, oferecendo uma performance superior em relação a abordagens baseadas em léxicos como VADER e LeIA, especialmente em idiomas como o português.

In [ ]:
avaliacoes_limpo.to_parquet('avaliacoes_otimizadas.parquet', index=False)

In [ ]:
!pip install pysentimiento

In [ ]:
from pysentimiento import create_analyzer
from tqdm import tqdm

tqdm.pandas()

df_ia = pd.read_parquet('avaliacoes_otimizadas.parquet')

analyzer = create_analyzer(task="sentiment", lang="pt")

def calcular_score_transformer(texto):
    if pd.isna(texto) or texto.strip() == "":
        return 0.0

    try:
        resultado = analyzer.predict(str(texto))
        probas = resultado.probas
        score = probas['POS'] - probas['NEG']
        return score
    except Exception as e:
        palavras = str(texto).split()[:150]
        texto_curto = ' '.join(palavras)
        resultado = analyzer.predict(texto_curto)
        probas = resultado.probas
        return probas['POS'] - probas['NEG']

def classificar_estrelas(score):
    if score >= 0.6: return 'Muito Positivo'
    elif score >= 0.2: return 'Positivo'
    elif score > -0.2: return 'Neutro'
    elif score > -0.6: return 'Negativo'
    else: return 'Muito Negativo'

df_ia['transformer_score'] = df_ia['review_text_clean'].progress_apply(calcular_score_transformer)
df_ia['sentimento_transformer'] = df_ia['transformer_score'].apply(classificar_estrelas)


display(df_ia[['review_score', 'review_text_clean', 'transformer_score', 'sentimento_transformer']].head(15))

df_ia.to_parquet('avaliacoes_com_transformer.parquet', index=False)

In [ ]:
contagem_transformer = df_ia['sentimento_transformer'].value_counts()

percentual_transformer = df_ia['sentimento_transformer'].value_counts(normalize=True) * 100

print("=== CONTAGEM ABSOLUTA (TRANSFORMER - PYSENTIMIENTO) ===")
print(contagem_transformer)
print("\n=== PERCENTAGEM (%) ===")
print(percentual_transformer.round(2))

#### Resultados do Transformer (Contagem e Percentual)

A distribuição de sentimentos do modelo Transformer apresenta um padrão interessante e que difere dos anteriores. Ele mostra uma alta concentração de avaliações 'Muito Positivas', seguido por 'Neutro' e 'Muito Negativo'. Isso sugere que o Transformer é mais decisivo em suas classificações extremas, identificando claramente avaliações com forte polaridade positiva ou negativa.

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))

ordem_sentimentos = ['Muito Negativo', 'Negativo', 'Neutro', 'Positivo', 'Muito Positivo']

ax = sns.countplot(
    data=df_ia,
    x='sentimento_transformer',
    order=ordem_sentimentos,
    palette='coolwarm_r'
)

plt.title('Distribuição de Sentimentos: IA Transformer (Pysentimiento)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Intensidade do Sentimento', fontsize=12)
plt.ylabel('Volume de Avaliações', fontsize=12)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom',
                fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

#### Visualização da Distribuição de Sentimentos (Transformer)

O gráfico de barras mostra uma distribuição mais polarizada, com as categorias 'Muito Positivo' e 'Muito Negativo' sendo mais proeminentes do que no LeIA, e muito mais do que no VADER. Isso indica que o modelo Transformer é mais agressivo na detecção de sentimentos fortes, classificando um número maior de avaliações em extremos, o que pode ser uma vantagem em contextos onde a intensidade do sentimento é crucial.

In [ ]:
df_ia = pd.read_parquet('avaliacoes_com_transformer.parquet')

matriz_confusao = pd.crosstab(
    df_ia['review_score'],
    df_ia['sentimento_transformer']
)

ordem = ['Muito Negativo', 'Negativo', 'Neutro', 'Positivo', 'Muito Positivo']
matriz_confusao = matriz_confusao.reindex(columns=ordem, fill_value=0)

plt.figure(figsize=(10, 7))
sns.heatmap(matriz_confusao, annot=True, fmt='d', cmap='Blues', cbar=False)

plt.title('Prova de Fogo: Estrelas Reais vs. Previsão da IA (Transformer)', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('O que a Inteligência Artificial Entendeu', fontsize=12)
plt.ylabel('Estrelas Dadas pelo Cliente (1 a 5)', fontsize=12)

plt.tight_layout()
plt.show()

#### Prova de Fogo: Estrelas Reais vs. Previsão do Transformer (Mapa de Calor)

O mapa de calor para o modelo Transformer demonstra uma correlação ainda mais refinada entre as estrelas dos clientes e as previsões do modelo. Há uma clara diagonal de correspondência, especialmente nas categorias extremas. Avaliações de 1 e 2 estrelas são fortemente associadas a 'Muito Negativo' e 'Negativo', e as de 4 e 5 estrelas a 'Positivo' e 'Muito Positivo'. A capacidade do Transformer de distinguir entre 'Positivo' e 'Muito Positivo', e 'Negativo' e 'Muito Negativo' de forma mais acentuada, o torna um candidato forte para análises detalhadas.

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(df_ia['transformer_score'], bins=20, kde=True, color='purple')

plt.title('Distribuição da Pontuação de Sentimento: IA Transformer', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Pontuação Matemática do Sentimento (-1.0 a +1.0)', fontsize=12)
plt.ylabel('Volume de Avaliações', fontsize=12)
plt.grid(axis='y', alpha=0.75)

plt.tight_layout()
plt.show()

#### Distribuição da Pontuação de Sentimento (IA Transformer)

O histograma da pontuação do Transformer mostra uma distribuição claramente bimodal, com picos significativos próximos a -1 e +1. Isso é um forte indicativo de que o modelo está classificando as avaliações com alta confiança em suas polaridades extremas, e menos frequentemente em uma neutralidade branda. A curva de densidade acentuada nos extremos demonstra a capacidade do Transformer de identificar e quantificar sentimentos fortes com precisão.

### 9. Comparação de Algoritmos: VADER vs LeIA vs Transformer

Nesta seção, realizamos uma "batalha" visual entre os três algoritmos de análise de sentimento (VADER, LeIA e Transformer) para entender suas performances relativas na classificação de sentimentos em português. O objetivo é destacar as diferenças nas distribuições de sentimento e identificar qual modelo se adapta melhor ao nosso conjunto de dados.

In [ ]:
df_batalha = pd.DataFrame({
    'VADER (Inglês)': avaliacoes_limpo['sentimento_vader_original'].values,
    'LeIA (PT-BR)': avaliacoes_limpo['sentimento_leia'].values,
    'Transformer (IA)': df_ia['sentimento_transformer'].values
})

df_derretido = df_batalha.melt(var_name='Modelo', value_name='Classificação')

ordem = ['Muito Negativo', 'Negativo', 'Neutro', 'Positivo', 'Muito Positivo']

plt.figure(figsize=(14, 7))

sns.countplot(
    data=df_derretido,
    x='Classificação',
    hue='Modelo',
    order=ordem,
    palette=['#e74c3c', '#2ecc71', '#9b59b6']
)

plt.title('Batalha de Algoritmos: VADER vs LeIA vs Transformer', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('O que o modelo entendeu do texto', fontsize=14)
plt.ylabel('Quantidade de Avaliações', fontsize=14)


plt.legend(title='Modelos', title_fontsize='13', fontsize='12')

plt.tight_layout()
plt.show()

#### Distribuição Comparativa de Sentimentos

O gráfico de barras agrupadas apresenta uma visão clara de como cada modelo classifica o sentimento das avaliações:

*   **VADER (Inglês):** Confirma a predominância de classificações 'Neutro', demonstrando sua limitação para textos em português. As categorias 'Muito Positivo' e 'Muito Negativo' são quase inexistentes.
*   **LeIA (PT-BR):** Mostra uma distribuição mais balanceada, com uma presença saudável de 'Positivo', 'Neutro' e 'Muito Positivo', além de uma parcela considerável de 'Negativo' e 'Muito Negativo'. Isso indica sua melhor adaptação ao idioma, mas ainda com uma tendência a suavizar os extremos.
*   **Transformer (IA):** Destaca-se pela polarização. Este modelo é o que mais atribui classificações 'Muito Positivo' e 'Muito Negativo', com menor incidência de 'Neutro' e 'Positivo'/ 'Negativo' moderados. Isso sugere que o Transformer é mais assertivo na detecção de sentimentos extremos, o que pode ser benéfico para identificar feedback de clientes muito satisfeitos ou muito insatisfeitos.

#### Conclusão da Comparação

A comparação visual reitera que modelos desenvolvidos ou adaptados para o português (LeIA e Transformer) superam significativamente o VADER. Enquanto o LeIA oferece uma visão mais equilibrada do espectro de sentimentos, o Transformer demonstra uma capacidade superior de identificar e quantificar sentimentos extremos, sendo uma ferramenta poderosa para cenários que exigem a detecção de opiniões muito fortes. A escolha do modelo ideal dependerá do objetivo específico da análise de sentimento.

# Merge

In [ ]:
df_final = pd.merge(df_clientes, df_pedidos, on='customer_id', how='inner')

df_final = pd.merge(df_final, df_item, on='order_id', how='inner')

df_final = pd.merge(df_final, df_produtos, on='product_id', how='inner')

df_final = pd.merge(df_final, df_pagamentos, on='order_id', how='inner')

df_final = pd.merge(df_final, df_vendedores, on='seller_id', how='inner')

df_final = pd.merge(df_final, df_avaliacoes, on='order_id', how='inner')

print(f"Merge concluído com sucesso! A super tabela agora tem {df_final.shape[0]} linhas e {df_final.shape[1]} colunas.")

display(df_final.head(6))

#### Detalhes do Mega Merge

Nesta etapa, consolidamos todas as informações relevantes em um único DataFrame chamado `df_final`. O processo envolveu a união sequencial de múltiplos DataFrames, utilizando chaves comuns (`customer_id`, `order_id`, `product_id`, `seller_id`) para garantir que os dados de clientes, pedidos, itens, produtos, pagamentos, vendedores e avaliações fossem integrados corretamente.

O merge foi realizado na seguinte ordem:
1.  `df_clientes` com `df_pedidos` (via `customer_id`)
2.  `df_final` resultante com `df_item` (via `order_id`)
3.  `df_final` resultante com `df_produtos` (via `product_id`)
4.  `df_final` resultante com `df_pagamentos` (via `order_id`)
5.  `df_final` resultante com `df_vendedores` (via `seller_id`)
6.  `df_final` resultante com `df_avaliacoes` (via `order_id`)

Ao final, obtivemos um DataFrame `df_final` com 117.329 linhas e 42 colunas, contendo uma visão completa de todo o ecossistema de dados para análises futuras.

# Ordenação

In [ ]:
amostra_vader = avaliacoes_limpo.head(2000)

data_for_sorting_vader = list(zip(amostra_vader['vader_score_original'], amostra_vader['review_text_clean']))


def bubble_sort_seguro_descendente(dados):
    lista_copia = list(dados)
    n = len(lista_copia)

    for i in range(n):
        for j in range(0, n - i - 1):

            if lista_copia[j][0] < lista_copia[j + 1][0]:
                lista_copia[j], lista_copia[j + 1] = lista_copia[j + 1], lista_copia[j]

    return lista_copia

print("\nOrdenando com Bubble Sort...")
textos_bubble_vader = bubble_sort_seguro_descendente(data_for_sorting_vader)

print("TOP 10 MAIS POSITIVOS (VADER - Bubble Sort):")
for score, review in textos_bubble_vader[:10]:
    print(f"({score:.2f}) {review}")

print("TOP 10 MAIS NEGATIVOS (VADER - Bubble Sort):")
for score, review in textos_bubble_vader[-10:]:
    print(f"({score:.2f}) {review}")

In [ ]:
def selection_sort_seguro_descendente(dados):
    lista_copia = list(dados)
    n = len(lista_copia)

    for i in range(n):
        max_idx = i
        for j in range(i + 1, n):
            if lista_copia[j][0] > lista_copia[max_idx][0]:
                max_idx = j

        lista_copia[i], lista_copia[max_idx] = lista_copia[max_idx], lista_copia[i]

    return lista_copia

print("Ordenando com Selection Sort...")
textos_selection_vader = selection_sort_seguro_descendente(data_for_sorting_vader)


print("TOP 10 MAIS POSITIVOS (VADER - Selection Sort):")
for score, review in textos_selection_vader[:10]:
    print(f"({score:.2f}) {review}")

print("TOP 10 MAIS NEGATIVOS (VADER - Selection Sort):")
for score, review in textos_selection_vader[-10:]:
    print(f"({score:.2f}) {review}")

In [ ]:
import sys

sys.setrecursionlimit(3000)


def quick_sort_seguro_descendente(dados):
    if len(dados) <= 1:
        return dados
    else:
        pivo = dados[0]
        maiores = [x for x in dados[1:] if x[0] >= pivo[0]]
        menores = [x for x in dados[1:] if x[0] < pivo[0]]

        return quick_sort_seguro_descendente(maiores) + [pivo] + quick_sort_seguro_descendente(menores)

print("Ordenando com Quick Sort (Avançado)...")
textos_quick_vader = quick_sort_seguro_descendente(data_for_sorting_vader)

print("TOP 10 MAIS POSITIVOS (VADER - Quick Sort):")
for score, review in textos_quick_vader[:10]:
    print(f"({score:.2f}) {review}")

print("TOP 10 MAIS NEGATIVOS (VADER - Quick Sort):")
for score, review in textos_quick_vader[-10:]:
    print(f"({score:.2f}) {review}")

In [ ]:
amostra_leia = avaliacoes_limpo.head(2000)

data_for_sorting_leia = list(zip(amostra_leia['pontuacao_sentimento_leia'], amostra_leia['review_text_clean']))

def selection_sort_seguro_descendente(dados):
    lista_copia = list(dados)
    n = len(lista_copia)

    for i in range(n):
        max_idx = i
        for j in range(i + 1, n):
            if lista_copia[j][0] > lista_copia[max_idx][0]:
                max_idx = j

        lista_copia[i], lista_copia[max_idx] = lista_copia[max_idx], lista_copia[i]

    return lista_copia

print("Ordenando com Selection Sort... Aguarde um instante.")
textos_selection_leia = selection_sort_seguro_descendente(data_for_sorting_leia)


print("TOP 10 MAIS POSITIVOS (LeIA - Selection Sort):")
for score, review in textos_selection_leia[:10]:
    print(f"(Score: {score:.2f}) {review}")


print("TOP 10 MAIS NEGATIVOS (LeIA - Selection Sort):")
for score, review in textos_selection_leia[-10:]:
    print(f"(Score: {score:.2f}) {review}")

In [ ]:
amostra_ia = df_ia.head(2000)

data_for_sorting_ia = list(zip(amostra_ia['transformer_score'], amostra_ia['review_text_clean']))

def selection_sort_seguro_descendente(dados):
    lista_copia = list(dados)
    n = len(lista_copia)

    for i in range(n):
        max_idx = i
        for j in range(i + 1, n):
            if lista_copia[j][0] > lista_copia[max_idx][0]:
                max_idx = j

        lista_copia[i], lista_copia[max_idx] = lista_copia[max_idx], lista_copia[i]

    return lista_copia

print("Ordenando com Selection Sort... Aguarde um instante.")
textos_selection_ia = selection_sort_seguro_descendente(data_for_sorting_ia)
print("TOP 10 MAIS POSITIVOS (Transformer - Selection Sort):")
print("="*70)
for score, review in textos_selection_ia[:10]:
    print(f"(Score: {score:.2f}) {review}")


print("TOP 10 MAIS NEGATIVOS (Transformer - Selection Sort):")
for score, review in textos_selection_ia[-10:]:
    print(f"(Score: {score:.2f}) {review}")

### Resumo da Ordenação: VADER, LeIA e Transformer

Nesta seção, aplicamos diferentes algoritmos de ordenação (Bubble Sort, Selection Sort e Quick Sort) para identificar as avaliações mais positivas e mais negativas de acordo com os scores de cada modelo de análise de sentimento (VADER, LeIA e Transformer).

#### VADER (Inglês) - Bubble Sort, Selection Sort, Quick Sort

Os três algoritmos de ordenação (Bubble Sort, Selection Sort e Quick Sort) foram aplicados aos scores do VADER. Como esperado, todos retornaram os mesmos top 10 comentários mais positivos e mais negativos, demonstrando a consistência dos algoritmos. As avaliações mais extremas (positivas e negativas) foram identificadas, embora o VADER, devido à sua natureza em inglês, mostre uma menor variação e polaridade nas pontuações em português.

#### LeIA (PT-BR) - Selection Sort

O Selection Sort foi utilizado para ordenar os scores do LeIA. Este algoritmo conseguiu extrair os top 10 comentários mais positivos e mais negativos com base na pontuação do LeIA, que é otimizado para o português. Os resultados exibem comentários que refletem sentimentos mais claros e alinhados com a realidade da língua portuguesa, com pontuações de polaridade mais acentuadas do que o VADER.

#### Transformer (Pysentimiento) - Selection Sort

Similarmente, o Selection Sort foi aplicado aos scores do modelo Transformer (Pysentimiento). As avaliações resultantes mostram uma polarização ainda mais forte, com comentários muito positivos e muito negativos recebendo scores próximos aos extremos de +1 e -1. Isso reforça a capacidade do Transformer de identificar sentimentos extremos com alta confiança, sendo muito eficaz para filtrar feedback de clientes extremamente satisfeitos ou insatisfeitos.

## Conclusão Geral: Modelos de Análise de Sentimento e Ordenação

Ao longo desta análise, exploramos o desempenho de três modelos distintos (VADER, LeIA e Transformer) na classificação de sentimentos em avaliações de clientes em português, complementado pela aplicação de algoritmos de ordenação para identificar as avaliações mais extremas.

### Análise de Sentimento:

1.  **VADER (Inglês):** Demonstrou ser o menos adequado para o português. Sua forte tendência a classificar a maioria dos textos como 'Neutro', e a fraca correlação com as `review_score` dos clientes, evidenciou a limitação de modelos desenvolvidos para um idioma diferente. Tanto nas contagens percentuais quanto nos mapas de calor e distribuições de score, o VADER não conseguiu capturar as nuances da língua portuguesa.

2.  **LeIA (PT-BR):** Ofereceu uma melhoria significativa, apresentando uma distribuição de sentimentos mais balanceada e uma correlação mais lógica com as `review_score`. Sendo otimizado para o português, o LeIA conseguiu distinguir melhor entre os sentimentos, com menos textos classificados como 'Neutro' e uma maior dispersão nas pontuações de sentimento, indicando maior sensibilidade linguística.

3.  **Transformer (Pysentimiento):** Destacou-se como o modelo de melhor desempenho. Sua arquitetura permitiu uma compreensão contextual mais profunda, resultando em classificações altamente polarizadas e uma forte correlação com as `review_score`, especialmente nas categorias 'Muito Positivo' e 'Muito Negativo'. O Transformer mostrou-se particularmente eficaz na identificação de sentimentos extremos, com uma distribuição bimodal clara em suas pontuações.

### Ordenação:

A aplicação de algoritmos de ordenação (Bubble Sort, Selection Sort e Quick Sort) nos scores de cada modelo foi crucial para extrair os *top 10* comentários mais positivos e negativos. Independentemente do algoritmo escolhido, os resultados foram consistentes para cada modelo, destacando a importância de ter um score numérico robusto para a ordenação eficaz.

*   **VADER:** Embora os algoritmos funcionassem, os comentários extremos identificados pelo VADER eram frequentemente menos expressivos em sua polaridade real, devido à sua baixa precisão para o português.
*   **LeIA:** A ordenação dos scores do LeIA resultou em uma seleção mais autêntica de comentários positivos e negativos, refletindo a maior acurácia do modelo para o idioma.
*   **Transformer:** A ordenação baseada nos scores do Transformer revelou os comentários com as polaridades mais fortes e confiáveis, confirmando sua capacidade de detectar os sentimentos mais extremos e claros expressos pelos clientes.

### Conclusão:

Para análise de sentimento em português, **o modelo Transformer (Pysentimiento) é a escolha superior**, oferecendo a maior precisão e capacidade de identificar sentimentos extremos, o que é valioso para ações como identificar clientes muito satisfeitos ou insatisfeitos. O LeIA é uma alternativa sólida para uma análise mais equilibrada, enquanto o VADER deve ser evitado para textos em português. A combinação de modelos de sentimento robustos com algoritmos de ordenação eficientes permite uma extração poderosa de insights a partir de grandes volumes de texto.

### Solução

Ao decorrer do projeto tivemos diversos problemas como comentarios vazios, comentarios com a ortografia errada, utilização de emojis com excesso e acentuação na qual atrapalha na hora da analise de sentimento. Como solução fizemos uma organização na tabela de avaliações para uma melhor analise. Tal organização foi feita uma limpeza de valores nulos, retirada de emojis e acentuação, já os problemas de ortografia não tinham como resolver manualmente